In [1]:
##############################################
##############################################
### TO GET ACCESS TO FILE IN GOOGLE COLAB ###
##############################################
##############################################

# To get access to the data files from github in Google Colab
!git clone https://github.com/math869c/graph_representation_st457.git

# set the folder to get access to the data
import os
os.chdir('/content') # to avoid error if rerun
os.chdir('./graph_representation_st457')

fatal: destination path 'graph_representation_st457' already exists and is not an empty directory.


In [2]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt
import torch.nn as nn
import torch
import torch.optim as optim
import json

# load from other folders
from helper_functions import *
from model_classes import *

# Load data for LSTM

In [ ]:
# Stock prices
open_prices_interp = pd.read_csv('data_folder/open_prices_interp.csv', index_col=0)
# into numpy
x = open_prices_interp.to_numpy()
tickers_with_data = open_prices_interp.columns

# Load graph data
with open("data_folder/firm_industry.json", "r") as f:
    firm_industry_dict = json.load(f)
A = np.load("data_folder/adjacency_matrix.npy")
adj_matrix = torch.tensor(A, dtype=torch.float32)

# make training data
batch_size =32
X_train, y_train, X_val, y_val, X_test, y_test, sc, train_loader, val_loader, test_loader = create_data(x,
                                                                                                        batch_size =batch_size,
                                                                                                        flatten_data = True, # Should be True for LSTM, False for GTC and GAT
                                                                                                        flatten_time_features=False # Should be True for GAT, False for LSTM and GTC
                                                                                                        )

torch.Size([984, 8, 2300])
torch.Size([984, 460])


# Load LSTM model

In [4]:
# this is j
output_size = y_train.shape[1]
input_size = X_train.shape[2]

model = LSTMRegressor(
    input_size=input_size,
    hidden_units=32,
    output_size=output_size,
    dropout_rate=0.2
)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters()) # just use standard learning rate for now


# Train LSTM Model

In [5]:
model_LSTM, history_metrics_LSTM, best_val_loss_LSTM = train_with_validation_LSTM(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    epochs=20,
    patience=10
)

test_loss_LSTM, _ = evaluate_LSTM(model_LSTM, test_loader, criterion)
print("Best validation loss:", best_val_loss_LSTM)
print("Test loss:", test_loss_LSTM)


Epoch 1/20 | Train Loss: 1.0122 Acc: 0.5005 | Val Loss: 2.6199 Val Acc: 0.5016
Epoch 2/20 | Train Loss: 1.0073 Acc: 0.5024 | Val Loss: 2.6161 Val Acc: 0.5033
Epoch 3/20 | Train Loss: 1.0037 Acc: 0.5081 | Val Loss: 2.6145 Val Acc: 0.5071
Epoch 4/20 | Train Loss: 0.9990 Acc: 0.5225 | Val Loss: 2.6160 Val Acc: 0.5102
Epoch 5/20 | Train Loss: 0.9906 Acc: 0.5347 | Val Loss: 2.6218 Val Acc: 0.5205
Epoch 6/20 | Train Loss: 0.9732 Acc: 0.5570 | Val Loss: 2.6410 Val Acc: 0.5109
Epoch 7/20 | Train Loss: 0.9609 Acc: 0.5709 | Val Loss: 2.6358 Val Acc: 0.5121
Epoch 8/20 | Train Loss: 0.9378 Acc: 0.5909 | Val Loss: 2.6702 Val Acc: 0.5021
Epoch 9/20 | Train Loss: 0.9355 Acc: 0.5939 | Val Loss: 2.6747 Val Acc: 0.5018
Epoch 10/20 | Train Loss: 0.9087 Acc: 0.6095 | Val Loss: 2.7162 Val Acc: 0.5037
Epoch 11/20 | Train Loss: 0.8881 Acc: 0.6193 | Val Loss: 2.7286 Val Acc: 0.5066
Epoch 12/20 | Train Loss: 0.8877 Acc: 0.6166 | Val Loss: 2.7185 Val Acc: 0.5020
Epoch 13/20 | Train Loss: 0.8592 Acc: 0.6304 | Va

# Evaluate LSTM

In [39]:
pred_scaled_LSTM = predict_final_model(model, test_loader)
# Manually inverse transform pred_scaled to undo the scaling
pred = pred_scaled_LSTM * sc.scale_[0] + sc.mean_[0]
real = y_test * sc.scale_[0] + sc.mean_[0]

In [40]:
# plot_results(y_test, real, open_prices_interp)